# WavqWise: EEG Classification with REAL Data
**Sense. Forecast. Alert.**

Real EEG data from MNE Sample Dataset (auditory vs visual evoked responses)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VK-Ant/wavqwise/blob/main/demos/notebooks/wavqwise_eeg_real_data.ipynb)

**Author:** [VK-Ant](https://github.com/VK-Ant)

In [ ]:
!pip install wavqwise mne scikit-learn seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from wavqwise import SignalPipeline
print('Ready')

## 1. Load REAL EEG Data (MNE Sample Dataset)

In [ ]:
import os
data_path = mne.datasets.sample.data_path()
raw_fname = os.path.join(data_path, 'MEG', 'sample', 'sample_audvis_filt-0-40_raw.fif')
raw = mne.io.read_raw_fif(raw_fname, preload=True, verbose=False)
raw.pick_types(meg=False, eeg=True, verbose=False)
print(f'Channels: {len(raw.ch_names)}')
print(f'Sample rate: {raw.info["sfreq"]} Hz')
print(f'Duration: {raw.times[-1]:.1f}s')

In [ ]:
# Extract epochs: Auditory vs Visual
events_fname = os.path.join(data_path, 'MEG', 'sample', 'sample_audvis_filt-0-40_raw-eve.fif')
events = mne.read_events(events_fname, verbose=False)
event_id = {'auditory': 1, 'visual': 3}
mask = np.isin(events[:, 2], [1, 3])
epochs = mne.Epochs(raw, events[mask], event_id=event_id, tmin=-0.2, tmax=0.5, baseline=(None, 0), preload=True, verbose=False)
epochs.drop_bad(verbose=False)
print(f'Auditory: {len(epochs["auditory"])} | Visual: {len(epochs["visual"])}')

## 2. WavqWise Band Analysis

In [ ]:
sig = SignalPipeline()
sig._data = epochs['auditory'].get_data()[0]  # First auditory epoch
sig._sample_rate = int(raw.info['sfreq'])
bands = sig.extract_bands(['delta','theta','alpha','beta','gamma'])
print('Auditory epoch band power:')
for name, power in bands.bands.items():
    print(f'  {name}: {power:.6f}')

## 3. Feature Extraction + Classification

In [ ]:
def extract_features(data, sfreq):
    band_defs = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,50)}
    feats = []
    for epoch in data:
        row = {}
        avg = epoch.mean(axis=0)
        freqs, psd = welch(avg, fs=sfreq, nperseg=min(int(sfreq), len(avg)))
        total = 0
        for name,(lo,hi) in band_defs.items():
            mask = (freqs>=lo)&(freqs<=hi)
            p = np.trapezoid(psd[mask], freqs[mask]) if hasattr(np,'trapezoid') else np.sum(psd[mask])
            row[name] = p; total += p
        for name in band_defs: row[f'{name}_rel'] = row[name]/max(total,1e-10)
        row['alpha_beta'] = row['alpha']/max(row['beta'],1e-10)
        row['theta_alpha'] = row['theta']/max(row['alpha'],1e-10)
        feats.append(row)
    return pd.DataFrame(feats)

X_aud = epochs['auditory'].get_data()
X_vis = epochs['visual'].get_data()
data = np.concatenate([X_aud, X_vis])
labels = np.concatenate([np.zeros(len(X_aud)), np.ones(len(X_vis))])
features = extract_features(data, raw.info['sfreq'])
print(f'Features: {features.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.3, random_state=42, stratify=labels)

for name, clf in [('RF', RandomForestClassifier(200, random_state=42)), ('GB', GradientBoostingClassifier(100, random_state=42)), ('SVM', SVC(random_state=42))]:
    sc = StandardScaler()
    clf.fit(sc.fit_transform(X_train), y_train)
    acc = accuracy_score(y_test, clf.predict(sc.transform(X_test)))
    print(f'{name}: {acc:.1%}')

## 4. Results

In [ ]:
clf = RandomForestClassifier(200, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Auditory','Visual']))

cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Aud','Vis'], yticklabels=['Aud','Vis'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
imp = pd.Series(clf.feature_importances_, index=features.columns).nlargest(8)
imp.plot(kind='barh', color='#2563eb', ax=axes[1])
axes[1].set_title('Feature Importance')
plt.suptitle('WavqWise EEG Classification - Real Data', fontweight='bold')
plt.tight_layout(); plt.show()

---
**WavqWise** - Sense. Forecast. Alert. | [GitHub](https://github.com/VK-Ant/wavqwise) | [PyPI](https://pypi.org/project/wavqwise/)